# 01 — Weight-Space Research Lab
### Transformer-first mathematical foundations with runnable demonstrations

**Primary architecture:** a small, transparent decoder-only PyTorch Transformer (2 layers, width 128, 4 attention heads, SwiGLU, RMSNorm, pre-norm residuals, tied token/output embeddings). It is randomly initialized and intended for controlled experiments—not a pretrained language model.

The notebook demonstrates architecture-aware symmetry, functional equivalence, parameter-space geometry, and compression concepts. A tiny MLP is no longer the centerpiece.


In [1]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
np.random.seed(7)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", DEVICE)

PyTorch: 2.10.0+cpu | device: cpu


## 1. Architecture-aware permutation symmetry

A Transformer head is a coordinated pathway. For this implementation, a head permutation must reorder the corresponding output-channel blocks of **Q, K, and V**, and the matching input-channel blocks of **O**. Reordering only one projection changes the function.

The same principle applies to SwiGLU: gate and up intermediate units must move together, with the matching columns of the down projection. These helpers are deliberately scoped to the explicit architecture defined below; arbitrary third-party checkpoints need architecture adapters.


In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        assert dim % heads == 0
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        b, t, d = x.shape
        def split_heads(z):
            return z.view(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (self.q_proj(x), self.k_proj(x), self.v_proj(x)))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        return self.o_proj(context)

class SwiGLU(nn.Module):
    def __init__(self, dim=128, hidden_dim=256):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim=128, heads=4, mlp_dim=256):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, mlp_dim)
    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

class MiniDecoderTransformer(nn.Module):
    """Small decoder-only LM for architecture-aware experiments; not pretrained."""
    def __init__(self, vocab_size=32, max_seq_len=16, dim=128, heads=4,
                 layers=2, mlp_dim=256, tie_embeddings=True):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, max_seq_len=max_seq_len,
                          dim=dim, heads=heads, layers=layers, mlp_dim=mlp_dim,
                          tie_embeddings=tie_embeddings)
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim) for _ in range(layers)
        ])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight
    def forward(self, input_ids):
        b, t = input_ids.shape
        if t > self.config['max_seq_len']:
            raise ValueError('Sequence exceeds configured max_seq_len')
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

def permute_attention_heads_(model, layer_index, permutation):
    """Function-preserving head permutation for this explicit MHA implementation."""
    attn = model.blocks[layer_index].attn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(attn.heads)):
        raise ValueError('permutation must contain each attention head exactly once')
    idx = torch.cat([torch.arange(int(h)*attn.head_dim, (int(h)+1)*attn.head_dim) for h in p]).to(attn.q_proj.weight.device)
    with torch.no_grad():
        for proj in (attn.q_proj, attn.k_proj, attn.v_proj):
            proj.weight.copy_(proj.weight[idx, :])
        attn.o_proj.weight.copy_(attn.o_proj.weight[:, idx])
    return model

def permute_swiglu_units_(model, layer_index, permutation):
    """Function-preserving permutation of the paired SwiGLU intermediate units."""
    ffn = model.blocks[layer_index].ffn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(ffn.gate_proj.out_features)):
        raise ValueError('permutation must contain every SwiGLU intermediate unit')
    with torch.no_grad():
        ffn.gate_proj.weight.copy_(ffn.gate_proj.weight[p])
        ffn.up_proj.weight.copy_(ffn.up_proj.weight[p])
        ffn.down_proj.weight.copy_(ffn.down_proj.weight[:, p])
    return model

@torch.no_grad()
def ablate_attention_head(model, layer_index, head_index):
    """Ablate one head's contribution by zeroing its input slice to O projection."""
    candidate = copy.deepcopy(model)
    attn = candidate.blocks[layer_index].attn
    if not 0 <= head_index < attn.heads:
        raise IndexError('head_index out of range')
    start, end = head_index * attn.head_dim, (head_index + 1) * attn.head_dim
    attn.o_proj.weight[:, start:end].zero_()
    return candidate

@torch.no_grad()
def ablate_swiglu_unit(model, layer_index, unit_index):
    candidate = copy.deepcopy(model)
    ffn = candidate.blocks[layer_index].ffn
    if not 0 <= unit_index < ffn.gate_proj.out_features:
        raise IndexError('unit_index out of range')
    ffn.gate_proj.weight[unit_index].zero_()
    ffn.up_proj.weight[unit_index].zero_()
    ffn.down_proj.weight[:, unit_index].zero_()
    return candidate

def next_token_loss(model, input_ids, target_ids):
    logits = model(input_ids)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))

model = MiniDecoderTransformer().to(DEVICE).eval()
tokens = torch.randint(0, model.config['vocab_size'], (8, 12), device=DEVICE)
with torch.no_grad():
    y_before = model(tokens)

head_permuted = copy.deepcopy(model)
permute_attention_heads_(head_permuted, layer_index=0, permutation=[2, 0, 3, 1])
with torch.no_grad():
    y_head = head_permuted(tokens)
head_error = (y_before - y_head).abs().max().item()
print('Attention-head permutation max output error:', head_error)
assert torch.allclose(y_before, y_head, atol=1e-4)

ffn_permuted = copy.deepcopy(model)
permute_swiglu_units_(ffn_permuted, 0, torch.randperm(model.blocks[0].ffn.gate_proj.out_features))
with torch.no_grad():
    y_ffn = ffn_permuted(tokens)
ffn_error = (y_before - y_ffn).abs().max().item()
print('SwiGLU permutation max output error:', ffn_error)
assert torch.allclose(y_before, y_ffn, atol=1e-4)
print('PASS: coordinated attention-head and SwiGLU permutations preserve this model function.')


Attention-head permutation max output error: 2.288818359375e-05
SwiGLU permutation max output error: 3.0517578125e-05
PASS: coordinated attention-head and SwiGLU permutations preserve this model function.


## 2. Invariance versus equivariance

- **Invariant:** transform the input and the output stays the same.
- **Equivariant:** transform the input and the output changes in the corresponding way.

Softmax is permutation-equivariant: reordering logits reorders probabilities in the same way. It does not make Softmax the origin of weight-space symmetry; it is simply a familiar example of the same mathematical property.

In [3]:
logits = torch.tensor([[2.0, 1.0, 0.0]])
p = torch.tensor([2, 0, 1])

probs = torch.softmax(logits, dim=-1)
permuted_probs = torch.softmax(logits[:, p], dim=-1)

print("Original probabilities: ", probs)
print("Permuted probabilities:", permuted_probs)
assert torch.allclose(permuted_probs, probs[:, p], atol=1e-7)
print("PASS: softmax is permutation-equivariant.")

Original probabilities:  tensor([[0.6652, 0.2447, 0.0900]])
Permuted probabilities: tensor([[0.0900, 0.6652, 0.2447]])
PASS: softmax is permutation-equivariant.


## 3. Functional equivalence is not raw weight similarity

A valid symmetry transformation can produce a large parameter-vector distance while preserving the model's outputs. We compare the original model with its head-permuted representation. This is why raw Euclidean weight distance is not, by itself, a reliable model-equivalence metric.


In [4]:
def flattened_parameters(model):
    return torch.cat([p.detach().reshape(-1).cpu() for p in model.parameters()])

raw_distance = torch.linalg.vector_norm(flattened_parameters(model) - flattened_parameters(head_permuted)).item()
with torch.no_grad():
    functional_error = (model(tokens) - head_permuted(tokens)).abs().max().item()
print(f'Raw parameter distance: {raw_distance:.6f}')
print(f'Maximum functional difference: {functional_error:.8f}')

Raw parameter distance: 18.487259
Maximum functional difference: 0.00002289


## 4. Variance across seeds: useful signal, not a deletion rule

A parameter's variance across independently trained models can reflect seed sensitivity, different valid representations, or genuinely different learned solutions. Before comparing coordinates, models may need alignment because neuron permutations make coordinate-wise comparisons misleading.

In [5]:
# Illustrative untrained initializations: variability is not evidence of learned functionality.
models = [MiniDecoderTransformer() for _ in range(8)]
stacked = torch.stack([m.blocks[0].attn.q_proj.weight.detach() for m in models])
coordinate_variance = stacked.var(dim=0, unbiased=False)
print('Mean Q-projection coordinate variance:', coordinate_variance.mean().item())
print('Variance range:', coordinate_variance.min().item(), coordinate_variance.max().item())

Mean Q-projection coordinate variance: 0.0022681730333715677
Variance range: 0.0001567123254062608 0.00566186523064971


## 5. Loss landscape: interpolate between equivalent Transformer parameter states

Even if two parameterizations implement the same function, a straight line between their raw parameter vectors need not remain functionally equivalent. This experiment interpolates between an original Transformer and a head-permuted equivalent. The result illustrates why parameter alignment matters before merging or interpolating models.


In [6]:
def set_flat_parameters_(model, flat):
    offset = 0
    with torch.no_grad():
        for parameter in model.parameters():
            n = parameter.numel()
            parameter.copy_(flat[offset:offset+n].view_as(parameter))
            offset += n
    assert offset == flat.numel()

base_model = MiniDecoderTransformer().eval()
equivalent_model = copy.deepcopy(base_model)
permute_attention_heads_(equivalent_model, 0, [1, 3, 0, 2])
theta_a, theta_b = flattened_parameters(base_model), flattened_parameters(equivalent_model)
probe_tokens = torch.randint(0, base_model.config['vocab_size'], (8, 12))
target_tokens = torch.randint(0, base_model.config['vocab_size'], (8, 12))
losses = []
for alpha in torch.linspace(0, 1, 11):
    candidate = copy.deepcopy(base_model)
    set_flat_parameters_(candidate, (1-alpha)*theta_a + alpha*theta_b)
    with torch.no_grad():
        losses.append(next_token_loss(candidate, probe_tokens, target_tokens).item())
print('Interpolation next-token losses:', [round(v, 5) for v in losses])

Interpolation next-token losses: [80.78606, 80.99445, 81.11657, 81.18278, 81.21462, 81.22487, 81.2176, 81.18814, 81.12299, 80.99955, 80.78606]


## 6. Kronecker factorization: a compact compression demonstration

For matrices, the Kronecker product of two smaller matrices can form a larger structured matrix. This is a useful compression concept, but the simple example below is **not** a reproduction of TQCompressor. The paper's permutation search/decomposition procedure requires its own faithful implementation and benchmark.

In [7]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[0.5, 0.0], [0.0, 0.5]])
W = torch.kron(A, B)

print("A shape:", tuple(A.shape), "B shape:", tuple(B.shape))
print("Kronecker matrix shape:", tuple(W.shape))
print("Dense parameters:", W.numel())
print("Factor parameters:", A.numel() + B.numel())
print(W)

A shape: (2, 2) B shape: (2, 2)
Kronecker matrix shape: (4, 4)
Dense parameters: 16
Factor parameters: 8
tensor([[0.5000, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.5000, 0.0000, 1.0000],
        [1.5000, 0.0000, 2.0000, 0.0000],
        [0.0000, 1.5000, 0.0000, 2.0000]])


## Takeaways

1. Transformer symmetries are coordinated transformations across related projections—not arbitrary tensor shuffles.
2. Invariance and equivariance describe how functions behave under transformations; they are not generic pruning algorithms.
3. Functionally equivalent models can be far apart in raw parameter coordinates.
4. Ablation measures an intervention's effect on a defined evaluation scope; it does not prove global irrelevance.
5. Architecture-aware mappings are prerequisites for safe Transformer manipulation.


## 7. Product relevance: why these foundations matter

The business objective is not to build a collection of academic demonstrations. These foundations support reusable capabilities in AI engineering and AI security products.

| Research concept | Product capability | Example customer question |
|---|---|---|
| Invariance / equivariance | Architecture-aware model comparison | Are these two checkpoints different in capability or only parameter ordering? |
| Weight-space representation | Model fingerprinting and retrieval | Which model artifacts are structurally similar? |
| Functional probes | Capability profiling | What does this model do on a declared probe suite? |
| Sensitivity / counterfactuals | Change-impact analysis | Which component changes actually affect this task? |
| Compression / factorization | Deployment optimization | Can this model be made smaller while preserving quality? |
| Weight generation | Experimental model synthesis | Can a candidate model be generated or refined from a learned weight distribution? |

**Product principle:** every scientific claim must map to a measurable output, a supported model family, and a stated limitation.

## 8. Research-to-demo map

These are research integrations to investigate, not claims that every method is production-ready.

- [NFN implementation](https://github.com/AllanYangZhou/nfn): permutation-equivariant processing of supported MLP/CNN weight structures.
- [WeightCLIP (ICML 2026)](https://proceedings.mlr.press/v306/asefaw26a.html): align model-weight and dataset representations for retrieval and model selection.
- [DeepWeightFlow (ICLR 2026)](https://proceedings.iclr.cc/paper_files/paper/2026/hash/e2f7407b62f152b7fe533fbc077fddb7-Abstract-Conference.html): generative modeling directly in weight space.
- [Finite Probes Suffice / HIDDENPROBE](https://www.microsoft.com/en-us/research/publication/finite-probes-suffice-identifiability-and-universality-for-weight-space-learning/): learn model properties from finite probe responses, including intermediate representations.
- [Weight Oracles](https://arxiv.org/abs/2610.07334): language-model-based diagnosis from raw weights; current scale limitations must remain visible.
- [Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/): early demonstrations of LLM reasoning over and editing model weights.

### Demo maturity labels
1. **Reproducible baseline:** code and evaluation can be run by us on a supported model.
2. **Research reproduction:** paper method is implemented with documented deviations.
3. **Prototype:** a limited demonstration inspired by the paper.
4. **Roadmap:** promising idea without a validated implementation.

Never present a prototype result as a validated research reproduction.

## 9. Knowledge graphs as a source of neural representations

This section extends the research lab with the mathematical bridge from qualified graph facts to tensors and trainable graph-derived computation.

A knowledge graph is represented as \(G=(V,E,R,X)\): entities \(V\), edges \(E\), relation types \(R\), and node features \(X\). A relation-specific adjacency matrix \(A_r\) encodes which entities participate in relation \(r\). A simple message-passing layer is:

\[
H^{(l+1)} = \sigma(\tilde A H^{(l)} W^{(l)} + b^{(l)})
\]

where \(\tilde A\) is a normalized adjacency matrix. The resulting hidden states are graph-conditioned representations—not a direct encoding that can be copied into arbitrary LLM token embeddings.

The runnable example below deliberately uses a tiny graph so that every matrix operation is inspectable.

In [8]:
# Tiny, deterministic knowledge graph -> matrix -> message passing demonstration
import torch
import torch.nn as nn
import torch.nn.functional as F

entities = ["Model", "Tokenizer", "Embedding"]
entity_to_id = {name: i for i, name in enumerate(entities)}
triples = [
    ("Model", "uses", "Tokenizer"),
    ("Tokenizer", "produces", "Embedding"),
]
relation_names = sorted({r for _, r, _ in triples})
relation_to_id = {name: i for i, name in enumerate(relation_names)}

A_by_relation = {}
for relation in relation_names:
    A = torch.zeros(len(entities), len(entities))
    for head, rel, tail in triples:
        if rel == relation:
            A[entity_to_id[head], entity_to_id[tail]] = 1.0
    A_by_relation[relation] = A

# Include self-loops, then symmetric degree normalization for this toy undirected
# propagation view. Directional relation matrices remain separately available above.
A = torch.zeros(len(entities), len(entities))
for rel_matrix in A_by_relation.values():
    A = A + rel_matrix + rel_matrix.T
A_hat = A + torch.eye(len(entities))
degree = A_hat.sum(dim=1)
D_inv_sqrt = torch.diag(degree.pow(-0.5))
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

# Small, explicit feature matrix: rows correspond to entities.
X = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
W_graph = nn.Parameter(torch.tensor([
    [0.8, 0.1],
    [0.2, 0.7],
    [0.4, 0.3],
]))
H = F.relu(A_norm @ X @ W_graph)

print("Entity order:", entities)
print("Relation matrices:", {k: tuple(v.shape) for k, v in A_by_relation.items()})
print("Normalized adjacency:\n", A_norm)
print("Graph-derived representation shape:", tuple(H.shape))
print("Trainable projection parameter count:", W_graph.numel())
assert H.shape == (3, 2)
assert torch.isfinite(H).all()

Entity order: ['Model', 'Tokenizer', 'Embedding']
Relation matrices: {'produces': (3, 3), 'uses': (3, 3)}
Normalized adjacency:
 tensor([[0.5000, 0.4082, 0.0000],
        [0.4082, 0.3333, 0.4082],
        [0.0000, 0.4082, 0.5000]])
Graph-derived representation shape: (3, 2)
Trainable projection parameter count: 6


### Research interpretation and boundaries

- Relation-specific adjacency matrices preserve relation identity; summing them is a simplification for this demonstration.
- The projection matrix is genuinely trainable, so it is a neural artifact that can be inspected and optimized.
- Graph embeddings are architecture- and objective-dependent. A projection or adapter is needed before a graph representation can condition a language model.
- The graph facts in this example are assumed inputs. In a real pipeline, provenance, confidence, source trust, temporal validity, and verification state must be retained before training.
- Tensor statistics describe numerical structure; behavioral tests are needed to establish whether a model learned a useful relationship.